In [35]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier,RandomForestClassifier,AdaBoostClassifier,GradientBoostingClassifier
import pandas as pd
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report

In [36]:
df=pd.read_csv("../data/os_scheduling_dataset.csv")
df.head()

,processes,num_processes,avg_burst_time,burst_time_variance,short_job_ratio,long_job_ratio,priority_variance,arrival_irregularity,burst_time_skewness,arrival_range,max_min_burst_ratio,best_algorithm
0,0:5:6:8|1:9:6:9|2:4:6:3|3:7:4:8|4:12:3:2|5:9:1...,8,3.875000,3.359375,0.500000,0.500000,5.437500,19.187500,-0.059000,15,6.000000,SJF
1,0:15:19:4|1:19:16:3|2:9:10:3,3,15.000000,14.000000,0.333333,0.666667,0.222222,16.888889,-0.381802,10,1.900000,FCFS
2,0:3:19:9|1:20:11:8|2:8:16:3|3:2:13:4|4:8:18:7|...,7,14.571429,9.959184,0.428571,0.571429,4.122449,48.693878,-0.067345,18,1.900000,FCFS
3,0:8:13:1|1:10:12:3|2:2:17:10,3,14.000000,4.666667,0.666667,0.333333,14.888889,11.555556,0.595170,8,1.416667,SJF
4,0:9:20:9|1:17:6:5|2:15:18:7|3:16:12:6|4:14:15:...,6,12.000000,44.333333,0.333333,0.666667,6.666667,7.555556,-0.447176,8,20.000000,Round Robin


In [37]:
df.columns

Index(['processes', 'num_processes', 'avg_burst_time', 'burst_time_variance',
       'short_job_ratio', 'long_job_ratio', 'priority_variance',
       'arrival_irregularity', 'burst_time_skewness', 'arrival_range',
       'max_min_burst_ratio', 'best_algorithm'],
      dtype='object')

In [38]:
X = df.drop(columns=["processes", "best_algorithm","long_job_ratio","arrival_range"],axis=1)
y = df["best_algorithm"]

In [39]:
print(X.shape)
print(y.value_counts())
print(X.isnull().sum())

(12000, 8)
best_algorithm
Round Robin    3130
FCFS           3068
SJF            2465
Priority       1800
SRTF           1537
Name: count, dtype: int64
num_processes           0
avg_burst_time          0
burst_time_variance     0
short_job_ratio         0
priority_variance       0
arrival_irregularity    0
burst_time_skewness     0
max_min_burst_ratio     0
dtype: int64


In [40]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y   # VERY IMPORTANT (keeps distribution same)
)

In [41]:
models = {
    "Decision Tree": DecisionTreeClassifier(
        max_depth=5,
        class_weight="balanced",
        random_state=42
    ),

    "Bagging": BaggingClassifier(
        estimator=DecisionTreeClassifier(max_depth=5),
        n_estimators=50,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=7,
        class_weight="balanced",
        random_state=42
    ),

    "AdaBoost": AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=2),
        n_estimators=50,
        learning_rate=0.5,
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    )
}

In [42]:
# from sklearn.metrics import accuracy_score, classification_report

# results = {}

# for name, model in models.items():
#     print(f"\n🔹 Training: {name}")

#     model.fit(X_train, y_train)
#     y_pred = model.predict(X_test)

#     acc = accuracy_score(y_test, y_pred)
#     results[name] = acc

#     print("Accuracy:", acc)
#     print(classification_report(y_test, y_pred))

In [43]:
# import pandas as pd

# results_df = pd.DataFrame(list(results.items()), columns=["Model", "Accuracy"])
# results_df = results_df.sort_values(by="Accuracy", ascending=False)

# print("\n📊 Model Comparison:")
# print(results_df)

In [44]:
# best_model_name = results_df.iloc[0]["Model"]
# print(f"\n🏆 Best Model: {best_model_name}")

In [45]:
dt_params = {
    "max_depth": [3, 5, 7, 10],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}
rf_params = {
    "n_estimators": [100, 150],
    "max_depth": [5, 8, 12],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}
from sklearn.tree import DecisionTreeClassifier

ada_params = {
    "n_estimators": [50, 100, 150],
    "learning_rate": [0.3, 0.5, 1.0],
    "estimator": [
        DecisionTreeClassifier(max_depth=1),
        DecisionTreeClassifier(max_depth=2)
    ]
}
gb_params = {
    "n_estimators": [100, 150],
    "learning_rate": [0.05, 0.08, 0.1],
    "max_depth": [3, 5],
    "subsample": [0.8, 1.0]
}


In [46]:
def tune_model(model, params, name):
    print(f"\n🔧 Tuning {name}...")

    grid = GridSearchCV(
        model,
        params,
        cv=3,
        scoring="accuracy",
        n_jobs=-1,
        verbose=1
    )

    grid.fit(X_train, y_train)

    print("Best Params:", grid.best_params_)
    print("Best CV Score:", grid.best_score_)

    best_model = grid.best_estimator_

    y_pred = best_model.predict(X_test)

    print("Test Accuracy:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred))

    return best_model

In [47]:
from sklearn.tree import DecisionTreeClassifier

dt_best = tune_model(
    DecisionTreeClassifier(class_weight="balanced", random_state=42),
    dt_params,
    "Decision Tree"
)


🔧 Tuning Decision Tree...
Fitting 3 folds for each of 36 candidates, totalling 108 fits
Best Params: {'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 2}
Best CV Score: 0.6711458333333332
Test Accuracy: 0.6695833333333333
              precision    recall  f1-score   support

        FCFS       0.74      0.42      0.54       614
    Priority       1.00      1.00      1.00       360
 Round Robin       0.84      0.42      0.56       626
         SJF       0.39      0.85      0.53       493
        SRTF       1.00      1.00      1.00       307

    accuracy                           0.67      2400
   macro avg       0.79      0.74      0.73      2400
weighted avg       0.77      0.67      0.67      2400



In [48]:
from sklearn.ensemble import RandomForestClassifier

rf_best = tune_model(
    RandomForestClassifier(class_weight="balanced", random_state=42),
    rf_params,
    "Random Forest"
)


🔧 Tuning Random Forest...
Fitting 3 folds for each of 24 candidates, totalling 72 fits
Best Params: {'max_depth': 8, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 100}
Best CV Score: 0.6733333333333333
Test Accuracy: 0.6641666666666667
              precision    recall  f1-score   support

        FCFS       0.61      0.51      0.55       614
    Priority       1.00      1.00      1.00       360
 Round Robin       0.84      0.40      0.54       626
         SJF       0.40      0.74      0.52       493
        SRTF       0.99      1.00      1.00       307

    accuracy                           0.66      2400
   macro avg       0.77      0.73      0.72      2400
weighted avg       0.73      0.66      0.67      2400



In [49]:
from sklearn.ensemble import AdaBoostClassifier

ada_best = tune_model(
    AdaBoostClassifier(random_state=42),
    ada_params,
    "AdaBoost"
)


🔧 Tuning AdaBoost...
Fitting 3 folds for each of 18 candidates, totalling 54 fits


c:\Users\SOHAM\anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


Best Params: {'estimator': DecisionTreeClassifier(max_depth=2), 'learning_rate': 0.3, 'n_estimators': 50}
Best CV Score: 0.6641666666666667
Test Accuracy: 0.65875
              precision    recall  f1-score   support

        FCFS       0.51      0.65      0.57       614
    Priority       1.00      1.00      1.00       360
 Round Robin       0.57      0.54      0.55       626
         SJF       0.50      0.37      0.42       493
        SRTF       1.00      1.00      1.00       307

    accuracy                           0.66      2400
   macro avg       0.72      0.71      0.71      2400
weighted avg       0.66      0.66      0.65      2400



In [50]:
from sklearn.ensemble import GradientBoostingClassifier

gb_best = tune_model(
    GradientBoostingClassifier(random_state=42),
    gb_params,
    "Gradient Boosting"
)


🔧 Tuning Gradient Boosting...
Fitting 3 folds for each of 24 candidates, totalling 72 fits
Best Params: {'learning_rate': 0.08, 'max_depth': 5, 'n_estimators': 150, 'subsample': 0.8}
Best CV Score: 0.6689583333333333
Test Accuracy: 0.6629166666666667
              precision    recall  f1-score   support

        FCFS       0.54      0.58      0.56       614
    Priority       1.00      1.00      1.00       360
 Round Robin       0.59      0.56      0.57       626
         SJF       0.46      0.44      0.45       493
        SRTF       1.00      1.00      1.00       307

    accuracy                           0.66      2400
   macro avg       0.72      0.72      0.72      2400
weighted avg       0.66      0.66      0.66      2400



In [51]:
models = {
    "Decision Tree": dt_best,
    "Random Forest": rf_best,
    "AdaBoost": ada_best,
    "Gradient Boosting": gb_best
}

results = []

for name, model in models.items():
    acc = accuracy_score(y_test, model.predict(X_test))
    results.append((name, acc))

results_df = pd.DataFrame(results, columns=["Model", "Accuracy"])
print(results_df.sort_values(by="Accuracy", ascending=False))

               Model  Accuracy
0      Decision Tree  0.669583
1      Random Forest  0.664167
3  Gradient Boosting  0.662917
2           AdaBoost  0.658750


In [52]:
import joblib

# Save model
joblib.dump(dt_best, "scheduler_model.pkl")

print("✅ Model saved successfully!")

✅ Model saved successfully!


In [53]:
feature_columns = X.columns.tolist()

joblib.dump(feature_columns, "feature_columns.pkl")

['feature_columns.pkl']